# Update the study lines' departments

Scrapes DTU's study database, sdb.dtu.dk, for the department of each study line and writes it to
`static_data.study_lines`, which `processing/new_proccess_data.ipynb` joins to the students for the
`line_department` column. `static_data.fetched` gets the time of the scrape, for the dashboard.

sdb.dtu.dk names no department a study line belongs to. The department here is that of the line's head of
studies (Studieleder), who normally comes from the department that runs it: a proxy.

Each year's page lists the BSc, BEng and MSc programmes, and those that no longer take students. The scrape
starts at this year and goes back to `FIRST_YEAR` (in `pis_lib/study_lines.py`), reading a programme's page
only for lines the newer years lack, so a line keeps its current department. `DIRECTION_LABELS` turns DTU's
programme names into the labels of `students.direction_DK`.

Run it on your own machine, after `uv sync --group courses`, when DTU publishes next year's study handbook,
and when the check below lists lines. The database is written directly.

In [ ]:
from datetime import date

import pandas as pd

from pis_lib import study_lines
from pis_lib.connect import get_pg_engine
from pis_lib.fetched import record_fetched

## Scrape

Takes about half a minute.

In [ ]:
found = study_lines.study_lines(study_lines.session(), date.today().year)
lines_df = pd.DataFrame(
    [(education, line, department) for (education, line), department in found.items()],
    columns=["education_EN", "direction_DK", "department"],
).sort_values(["education_EN", "direction_DK"])
print(len(lines_df), "study lines")
assert len(lines_df) >= 50, f"only {len(lines_df)} study lines: has sdb.dtu.dk changed its pages?"

## Check

The study lines of `static_data.students` without a department here, by how many students follow each: their
visits get `Unknown`. Usually a programme whose name `DIRECTION_LABELS` doesn't turn into the line's label: add
it there and scrape again.

In [ ]:
pg = get_pg_engine()
students = pd.read_sql(
    'SELECT "education_EN", "direction_DK", count(*) AS students FROM static_data.students'
    ' WHERE "direction_DK" IS NOT NULL GROUP BY 1, 2', pg,
)
uncovered = students.merge(lines_df, on=["education_EN", "direction_DK"], how="left")
print(uncovered[uncovered["department"].isna()].drop(columns="department")
      .sort_values("students", ascending=False).to_string(index=False))

## Write to `static_data`

> **Warning:** `if_exists="replace"` drops and recreates the table on every run.

In [ ]:
with pg.begin() as db:
    lines_df.to_sql("study_lines", db, schema="static_data", if_exists="replace", index=False)
    record_fetched(db, "study_lines")